In [ ]:
import re
from pathlib import Path

import fitz  # PyMuPDF
import pandas as pd


# -----------------------------
# Regexy
# -----------------------------
ROOM_RE = re.compile(r"^(?P<kod>[0-9]{3}[a-z]?/G\.\d{2})\b", re.IGNORECASE)
UNIT_ONLY_RE = re.compile(r"^(?P<unit>[0-9]{3}[a-z]?/G)\b$", re.IGNORECASE)

TITLE_TAB = ["Tabulka místností", "Tabulka mistnosti"]
TITLE_VYP = ["Výpis překladů", "Vypis prekladu", "Výpis prekladů", "Vypis prekladu"]

def clean_leading_index(s: str) -> str:
    # odstraňuje číslování řádku tabulky typu: "3 201/G.01 ..."
    return re.sub(r"^\d+\.?\s+", "", s.strip())

def group_words_to_lines(words, y_tol=2.0):
    """
    words: (x0, y0, x1, y1, text, block, line, word)
    skládá slova do řádků podle y0 (tolerance y_tol)
    """
    words = sorted(words, key=lambda w: (w[1], w[0]))
    lines = []
    cur = []
    cur_y = None

    for x0, y0, x1, y1, txt, *_ in words:
        if cur_y is None:
            cur_y = y0
            cur = [(x0, txt)]
            continue
        if abs(y0 - cur_y) <= y_tol:
            cur.append((x0, txt))
        else:
            cur_sorted = sorted(cur, key=lambda t: t[0])
            line = " ".join(t for _, t in cur_sorted).strip()
            if line:
                lines.append(line)
            cur_y = y0
            cur = [(x0, txt)]

    if cur:
        cur_sorted = sorted(cur, key=lambda t: t[0])
        line = " ".join(t for _, t in cur_sorted).strip()
        if line:
            lines.append(line)

    return [clean_leading_index(l) for l in lines if l.strip()]

def parse_room_line(line: str):
    """
    Parsuje řádek místnosti:
    201/G.01 chodba 5,5 m² 2 655 mm ... DL-... P-... S-... OM-...
    """
    raw = line
    line = clean_leading_index(line)

    m = ROOM_RE.match(line)
    if not m:
        return None

    kod = m.group("kod")
    jednotka = kod.split(".")[0]

    area_m = re.search(r"\b(\d{1,3}(?:[.,]\d+)?)\s*m²\b", line)
    h_m    = re.search(r"\b([0-9]{1,2}(?:\s[0-9]{3})?)\s*mm\b", line)
    if not area_m or not h_m:
        return None

    plocha = float(area_m.group(1).replace(",", "."))
    vyska  = int(h_m.group(1).replace(" ", ""))

    name = line[m.end():area_m.start()].strip()
    name = re.sub(r"\s+", " ", name)

    dl = re.search(r"\bDL-[0-9]+[a-z]?\b(?:\s*,\s*DL-[0-9]+[a-z]?\b)*", line, re.IGNORECASE)
    p  = re.search(r"\bP-[0-9]+[a-z]?\b(?:\s*,\s*P-[0-9]+[a-z]?\b)*", line, re.IGNORECASE)
    s  = re.search(r"\bS-[0-9]+[a-z]?\b(?:\s*,\s*S-[0-9]+[a-z]?\b)*", line, re.IGNORECASE)
    om = re.search(r"\bOM-[0-9]+[a-z]?\b(?:\s*,\s*OM-[0-9]+[a-z]?\b)*", line, re.IGNORECASE)

    return {
        "Jednotka": jednotka,
        "Kód": kod,
        "Název": name,
        "Plocha_m2": plocha,
        "Světlá_výška_mm": vyska,
        "Skladba_podlahy_DL": dl.group(0) if dl else None,
        "Kód_P": p.group(0) if p else None,
        "Kód_S": s.group(0) if s else None,
        "Kód_OM": om.group(0) if om else None,
        "RAW": raw,
    }

def find_title_y(page, candidates):
    """
    najde y souřadnici (horní) pro první výskyt některého titulku.
    vrací y0 nebo None
    """
    for t in candidates:
        rects = page.search_for(t)
        if rects:
            # vezmeme ten nejvíc vpravo (tabulky jsou vpravo)
            rect = sorted(rects, key=lambda r: r.x0, reverse=True)[0]
            return rect.y0
    return None

def extract_words_lines(page, clip, y_tol=2.0):
    words = page.get_text("words", clip=clip)
    return group_words_to_lines(words, y_tol=y_tol)

def extract_pdf(pdf_path: Path, out_xlsx: Path, page_index=0):
    doc = fitz.open(str(pdf_path))
    page = doc[page_index]
    pr = page.rect
    w, h = pr.width, pr.height

    # 1) Pravý panel (tabulky jsou vždy vpravo)
    right_panel = fitz.Rect(w * 0.58, 0, w, h)

    # 2) Najdi přibližně y pozice nadpisů (pokud to selže, jedeme fallback)
    y_tab = find_title_y(page, TITLE_TAB)
    y_vyp = find_title_y(page, TITLE_VYP)

    # 3) Vymez oblasti tabulek v rámci pravého panelu
    #    (přidáme rozumné paddingy)
    # --- robustní fallback: nepředpokládáme, že nadpisy existují / jsou stejné ---
    # Primární (pokud se najdou nadpisy): použijeme je.
    # Sekundární: pevné poměry pravého panelu.

    pad_top = 10
    pad_mid = 10

    if y_tab is not None and y_vyp is not None and (y_vyp - y_tab) > (h * 0.10):
        # nadpisy dávají smysl
        tab_clip = fitz.Rect(right_panel.x0, max(0, y_tab - pad_top), right_panel.x1, min(h, y_vyp - pad_mid))
        vyp_clip = fitz.Rect(right_panel.x0, max(0, y_vyp - pad_top), right_panel.x1, h)
    else:
        # fallback: horní část pravého panelu = místnosti, dolní = překlady
        split_y = h * 0.68
        tab_clip = fitz.Rect(right_panel.x0, 0, right_panel.x1, split_y)
        vyp_clip = fitz.Rect(right_panel.x0, split_y, right_panel.x1, h)

    # 4) Tabulka místností je často rozdělená na dva vnitřní sloupce => rozřízneme tab_clip na poloviny
    midx = tab_clip.x0 + (tab_clip.width / 2.0)
    tab_left  = fitz.Rect(tab_clip.x0, tab_clip.y0, midx, tab_clip.y1)
    tab_right = fitz.Rect(midx, tab_clip.y0, tab_clip.x1, tab_clip.y1)

    lines_left  = extract_words_lines(page, tab_left)
    lines_right = extract_words_lines(page, tab_right)

    # 5) Parse místností
    rooms = []
    current_unit = None

    for ln in (lines_left + lines_right):
        ln = clean_leading_index(ln)

        if re.search(r"Společné\s+prostory", ln, re.IGNORECASE):
            current_unit = "Společné prostory"
            continue

        um = UNIT_ONLY_RE.match(ln)
        if um:
            current_unit = um.group("unit")
            continue

        rec = parse_room_line(ln)
        if rec:
            # když máme detekovanou hlavičku jednotky, použij ji jako jistější zdroj
            if current_unit and current_unit != "Společné prostory":
                rec["Jednotka"] = current_unit
            rooms.append(rec)

    df_rooms = pd.DataFrame(rooms)

    # společné prostory: v těchto výkresech typicky 100/G a 200/G
    df_common = df_rooms[df_rooms["Jednotka"].isin(["100/G", "200/G"])].copy()
    df_rooms2 = df_rooms[~df_rooms["Jednotka"].isin(["100/G", "200/G"])].copy()

    # 6) Výpis překladů
    vypis_lines = extract_words_lines(page, vyp_clip)
    vypis = []

    for ln in vypis_lines:
        ln = clean_leading_index(ln)
        if not re.match(r"^p\d+\w?\b", ln, re.IGNORECASE):
            continue

        m = re.match(r"^(p\d+\w?)\s+(.*)$", ln, re.IGNORECASE)
        ozn = m.group(1)
        rest = m.group(2)

        nums = re.findall(r"\b\d+\b", rest)
        width  = int(nums[0]) if len(nums) >= 1 else None
        height = int(nums[1]) if len(nums) >= 2 else None
        length = int(nums[2]) if len(nums) >= 3 else None
        count  = int(nums[-1]) if nums else None
        # pokud nemáme aspoň 3 rozměry, není to položka překladu -> přeskoč
        if width is None or height is None or length is None:
            continue



        pop = rest
        if width and height and length:
            pop = re.sub(rf"^{width}\s+{height}\s+{length}\s+", "", pop)
        if count is not None:
            pop = re.sub(rf"\s+{count}\s*$", "", pop)

        vypis.append({
            "Označení": ozn,
            "Šířka_mm": width,
            "Výška_mm": height,
            "Délka_mm": length,
            "Popis": pop.strip(),
            "Počet_ks": count,
            "RAW": ln,
        })

    df_vypis = pd.DataFrame(vypis)

    # 7) Export
    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as w:
        df_rooms2.to_excel(w, index=False, sheet_name="Tabulka_mistnosti")
        df_common.to_excel(w, index=False, sheet_name="Spolecne_prostory")
        df_vypis.to_excel(w, index=False, sheet_name="Vypis_prekladu")
        pd.DataFrame({
            "Zdroj": [pdf_path.name],
            "PageIndex": [page_index],
            "RightPanel": [f"{right_panel}"],
            "TabClip": [f"{tab_clip}"],
            "VypClip": [f"{vyp_clip}"],
        }).to_excel(w, index=False, sheet_name="Info")

    doc.close()


if __name__ == "__main__":
    # 1) jeden soubor:
    # python extract_tables_rightpanel.py "NO_DPS_SO07_D.01.08_Půdorys 1.NP_B2.pdf"
    import sys

    if len(sys.argv) < 2:
        raise SystemExit('Zadej cestu k PDF, např.: python extract_tables_rightpanel.py "soubor.pdf"')

    pdf = Path(sys.argv[1])
    out = pdf.with_name(pdf.stem + "__tabulky.xlsx")
    extract_pdf(pdf, out_xlsx=out, page_index=0)
    print(f"OK: {out}")


In [10]:
from pathlib import Path
import pandas as pd


FILES = [
    "NO_DPS_SO07_D.01.04_Půdorys 4.PP_B.pdf",
    "NO_DPS_SO07_D.01.05_Půdorys 3.PP_B.pdf",
    "NO_DPS_SO07_D.01.06_Půdorys 2.PP_B.pdf",
    "NO_DPS_SO07_D.01.07_Půdorys 1.PP_B2.pdf",
    "NO_DPS_SO07_D.01.08_Půdorys 1.NP_B2.pdf",
    "NO_DPS_SO07_D.01.09_Půdorys 2.NP_B2.pdf",
    "NO_DPS_SO07_D.01.10_Půdorys 3.NP_B2.pdf",
    "NO_DPS_SO07_D.01.11_Půdorys 4.NP_B2.pdf",
    "NO_DPS_SO07_D.01.12_Půdorys 5.NP_B2.pdf",
    "NO_DPS_SO07_D.01.13_Půdorys 6.NP_B2.pdf",
    "NO_DPS_SO07_D.01.14_Půdorys 7.NP_B2.pdf",
    "NO_DPS_SO07_D.01.15_Půdorys 8.NP_B2.pdf",
    "NO_DPS_SO07_D.01.16_Půdorys 9.NP_B2.pdf",
    "NO_DPS_SO07_D.01.17_Půdorys 10.NP_B2.pdf",
    "NO_DPS_SO07_D.01.18_Půdorys 11.NP_B2.pdf",
    "NO_DPS_SO07_D.01.19_Půdorys 12.NP_B2.pdf",
]


def floor_from_filename(name: str) -> str:
    """
    Vytáhne text podlaží z názvu souboru, např. '1.NP', '4.PP'
    """
    import re
    m = re.search(r"(\d+\.(?:NP|PP))", name)
    return m.group(1) if m else ""


def safe_read_sheet(xlsx_path: Path, sheet: str):
    try:
        return pd.read_excel(xlsx_path, sheet_name=sheet)
    except Exception:
        return None


def main():
    base_dir = Path(".").resolve()

    all_rooms = []
    all_common = []
    all_vypis = []
    log_rows = []

    for fname in FILES:
        pdf = base_dir / fname
        if not pdf.exists():
            log_rows.append({"file": fname, "status": "SKIP", "reason": "file not found"})
            continue

        floor = floor_from_filename(pdf.name)

        try:
            # 1) vyrob dočasný xlsx pro jeden pdf
            tmp_xlsx = pdf.with_name(pdf.stem + "__tabulky.xlsx")
            extract_pdf(pdf, out_xlsx=tmp_xlsx, page_index=0)

            # 2) načti listy a přidej metadata
            df_rooms = safe_read_sheet(tmp_xlsx, "Tabulka_mistnosti")
            df_common = safe_read_sheet(tmp_xlsx, "Spolecne_prostory")
            df_vypis = safe_read_sheet(tmp_xlsx, "Vypis_prekladu")

            if df_rooms is not None and not df_rooms.empty:
                df_rooms.insert(0, "Zdroj_PDF", pdf.name)
                df_rooms.insert(1, "Podlazi", floor)
                all_rooms.append(df_rooms)

            if df_common is not None and not df_common.empty:
                df_common.insert(0, "Zdroj_PDF", pdf.name)
                df_common.insert(1, "Podlazi", floor)
                all_common.append(df_common)

            if df_vypis is not None and not df_vypis.empty:
                df_vypis.insert(0, "Zdroj_PDF", pdf.name)
                df_vypis.insert(1, "Podlazi", floor)
                all_vypis.append(df_vypis)

            log_rows.append({"file": pdf.name, "status": "OK", "reason": ""})

        except Exception as e:
            log_rows.append({"file": pdf.name, "status": "FAIL", "reason": f"{type(e).__name__}: {e}"})

    # 3) sloučení
    df_rooms_all = pd.concat(all_rooms, ignore_index=True) if all_rooms else pd.DataFrame()
    df_common_all = pd.concat(all_common, ignore_index=True) if all_common else pd.DataFrame()
    df_vypis_all = pd.concat(all_vypis, ignore_index=True) if all_vypis else pd.DataFrame()
    df_log = pd.DataFrame(log_rows)
    df_log[df_log["status"] == "FAIL"][["file", "reason"]]


    # 4) uložení do jednoho XLSX
    out = base_dir / "SO07_B2__TABULKY_ALL.xlsx"
    with pd.ExcelWriter(out, engine="openpyxl") as w:
        df_rooms_all.to_excel(w, index=False, sheet_name="Mistnosti_ALL")
        df_common_all.to_excel(w, index=False, sheet_name="Spolecne_ALL")
        df_vypis_all.to_excel(w, index=False, sheet_name="Preklady_ALL")
        df_log.to_excel(w, index=False, sheet_name="Log")

    print(f"Hotovo: {out}")
    print(df_log["status"].value_counts(dropna=False))


if __name__ == "__main__":
    main()


Hotovo: E:\DATA_ANALYSIS\SO07_B2__TABULKY_ALL.xlsx
status
FAIL    12
OK       4
Name: count, dtype: int64
